# Advanced Certification Program in Computational Data Science

##  A program by IISc and TalentSprint

### Notebook: Market Basket Analysis
Note: This is supplementary reading material provided with the Market Basket Analysis Mini-Project

## Learning Objectives

At the end of the experiment, you will be able to :

* extract summary level insight from a given customer dataset

* handle the missing data and identify the underlying pattern or structure

* identify customer segments based on the overall buying behaviour


## Dataset

The dataset chosen for this mini project is Online Retail dataset. It is a transnational data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.

The dataset contains 541909 records, and each record is made up of 8 fields.

To know more about the dataset : [click here](https://cdn.iisc.talentsprint.com/CDS/Assignments/Module6/M6_Supplementary_NB_2_Market_Basket_Analysis_online%20retail%20dataset.pdf)

## Information

#### Market Basket Analysis

Market Basket Analysis is one of the key techniques used by the large retailers that uncovers associations between items by looking for combinations of items that occur together frequently in transactions. In other words, it allows the retailers to identify relationships between the items that people buy.

Association Rules is widely used to analyze retail basket or transaction data, is intended to identify strong rules discovered in transaction data using some measures of interestingness, based on the concept of strong rules.

##### Example of Association Rules

* Assume there are 100 customers
* 10 out of them bought milk, 8 bought butter and 6 bought both of them.
* bought milk => bought butter
* Support = P(Milk & Butter) = 6/100 = 0.06
* confidence = support/P(Butter) = 0.06/0.08 = 0.75
* lift = confidence/P(Milk) = 0.75/0.10 = 7.5


**Note:** In practice, a rule needs a support of several hundred transactions before it can be considered statistically significant, and datasets often contain thousands or millions of transactions.


To know more about the **market basket analysis** click [here](https://cdn.iisc.talentsprint.com/CDS/Assignments/Module6/market%20basket%20analysis%20definitions%20examples.pdf)




### Import required packages

In [1]:
import numpy as np  # Importing Numpy Package
from scipy import stats
import pandas as pd  # Importing Pandas Package under name pd
from mlxtend.frequent_patterns import (
    apriori,
    association_rules,
)  # Importing apriori and association rules from mlxtend package


To know about **mlxtend.frequent_patterns** click [here](http://rasbt.github.io/mlxtend/api_subpackages/mlxtend.frequent_patterns/)

## Data Wrangling

In [2]:
# @title Download the data
from utility import download_and_unzip

download_and_unzip(
    filename="Online_Retail.xlsx",
    url="https://cdn.iiith.talentsprint.com/CDS/Datasets/Online_Retail.xlsx",
)
# !wget -qq https://cdn.iiith.talentsprint.com/CDS/Datasets/Online_Retail.xlsx
print("Data downloaded successfully")


Data downloaded successfully


#### Loading the data

In [3]:
data = pd.read_excel("Online_Retail.xlsx")  # Loading the data


To know more about the **read_excel** function click [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_excel.html)

In [4]:
data.head()  # Checking for the first five rows from the dataset


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
data.tail()  # Checking for the last five rows from the dataset


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [6]:
data.shape  # Checking for the number of rows and columns in the dataset


(541909, 8)

In [7]:
data.columns  # Checking for the columns in the dataset


Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [8]:
data.dtypes  # Checking for the types of variables in the dataframe


InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

## Exploratory Data Analysis

### Data Pre-processing

Checking for the duplicate data using **duplicated** funtion.

To know about duplicate function click [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.duplicated.html)

In [9]:
data.duplicated()


0         False
1         False
2         False
3         False
4         False
          ...  
541904    False
541905    False
541906    False
541907    False
541908    False
Length: 541909, dtype: bool

We can see that there is a lot of redundant data. So, let's handle the redundant data by dropping them using the **drop_duplicates** function.

To know more about the **drop_duplicates** click [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop_duplicates.html)

In [10]:
data.drop_duplicates(inplace=True)  # Handling the redundant data by dropping them
data.shape  # Checking for the shape of the data after dropping the redundant data


(536641, 8)

In the dataset, we can see that most Invoices appear as normal transactions with positive quantity and prices, but there are some prefixed with "C" or "A" which denote different transaction types. Invoice starting with C represents cancelled order and A represents the Adjusted.

Now let's identify such data and handle them by checking the negative values in Quantity column for all cancelled orders

In [11]:
(
    len(data[data.InvoiceNo.str[0] == "C"]),
    len(data[data.Quantity < 1]),
)  # Checking for the data


(9251, 10587)

Dropping the records containing the Cancelled orders.

To know about how to subset a pandas dataframe click [here](https://cdn.iisc.talentsprint.com/CDS/Assignments/Module6/M6_Supplementary_NB_2_Market_Basket_Analysis_How%20To%20Filter%20Pandas%20Dataframe.pdf)

In [12]:
data = data[~(data.InvoiceNo.str[0] == "C")]
data.shape  # Checking for the shape of the dataset after dropping the records which contain cancelled orders


(527390, 8)

### Descriptive statistics

Let's describe the statistics of the data using **describe().**

To know about the **describe** function click [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html)

In [13]:
data.describe()


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,527390.000000,527390,527390.000000,392732.000000
mean,10.311272,2011-07-04 12:21:06.631866112,3.861939,15287.734822
min,-9600.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:49:00,1.250000,13955.000000
50%,3.000000,2011-07-19 15:55:00,2.080000,15150.000000
75%,11.000000,2011-10-19 10:56:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,13541.330000,18287.000000
std,160.367285,NaN,41.963759,1713.567773


Checking for the empty records(null values) using **isna** function.

To know more about the isna function click [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isna.html)

In [14]:
data.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [15]:
data.isna().sum()


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     134658
Country             0
dtype: int64

let's drop the empty records using **dropna** function.

To know more about the dropna function click [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html)

In [16]:
data.dropna(inplace=True)
data.shape  # Checking for the shape of the data after dropping the nan values


(392732, 8)

From the dataset, we can see that some of the transactions based on the `StockCode` variable are not actually products, but representing the costs or fees regarding to the post or bank or other tansactions.

Let's assume that the transaction with `'POST' 'PADS' 'M' 'DOT' 'C2' 'BANK CHARGES'` as their `StockCodes` are irrelevant transactions.

In [17]:
Irrelevant = data["StockCode"].astype("str").unique()  # Finding the irrelevant data
Irrelevant.sort()  # Sorting the data
Irrelevant


array(['10002', '10080', '10120', ..., 'M', 'PADS', 'POST'],
      shape=(3665,), dtype=object)

In [18]:
print("Irrelevant Transactions: \n", Irrelevant[::-1][:100])
print(data.shape)  # Checking for the shape of the data
data = data[
    ~(data["StockCode"].isin(["POST", "PADS", "M", "DOT", "C2", "BANK CHARGES"]))
]  # Dropping irrelevant data
print(data.shape)  # Checking for the shape of the data after removing irrelevant data


Irrelevant Transactions: 
 ['POST' 'PADS' 'M' 'DOT' 'C2' 'BANK CHARGES' '90214Z' '90214Y' '90214W'
 '90214V' '90214U' '90214T' '90214S' '90214R' '90214P' '90214O' '90214N'
 '90214M' '90214L' '90214K' '90214J' '90214I' '90214H' '90214G' '90214F'
 '90214E' '90214D' '90214C' '90214B' '90214A' '90212C' '90212B' '90211B'
 '90211A' '90210D' '90210C' '90210B' '90210A' '90209C' '90209B' '90209A'
 '90208' '90206C' '90206A' '90205C' '90205A' '90204' '90202D' '90202C'
 '90202B' '90202A' '90201D' '90201C' '90201B' '90201A' '90200E' '90200D'
 '90200C' '90200B' '90200A' '90199D' '90199C' '90199B' '90199A' '90198B'
 '90198A' '90197B' '90196B' '90196A' '90195B' '90195A' '90194' '90192'
 '90191' '90190C' '90190B' '90190A' '90189A' '90188' '90187B' '90186B'
 '90186A' '90185D' '90185C' '90185B' '90185A' '90184C' '90184B' '90184A'
 '90183C' '90183A' '90182C' '90181A' '90180B' '90180A' '90179C' '90179A'
 '90178B' '90178A' '90177E']
(392732, 8)
(391183, 8)


We can see that there are outliers in the UnitPrice and Quantity Variables. Let's handle them by calculating the z-score.

To know about how to handle outliers click [here](https://cdn.iisc.talentsprint.com/CDS/Assignments/Module6/M6_Supplementary_NB_2_Market_Basket_Analysis_How%20To%20Remove%20outliers.pdf)

In [19]:
data = data[
    (np.abs(stats.zscore(data["UnitPrice"])) < 3)
    & (np.abs(stats.zscore(data["Quantity"])) < 5)
]
data.shape  # Checking for the shape of the data


(388722, 8)

We need to consolidate the items into 1 transaction per row with each product 1 hot encoded. For the sake of keeping the data set small, We will be looking at sales for France to apply association rule by grouping the Invoice and Quantity variables.

In [20]:
basket_France = (
    data[data["Country"] == "France"]
    .groupby(["InvoiceNo", "Description"])["Quantity"]
    .sum()
    .unstack()
    .reset_index()
    .fillna(0)
    .set_index("InvoiceNo")
)
basket_France


Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TRELLIS COAT RACK,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,WRAP SUKI AND FRIENDS,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,ZINC STAR T-LIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536370,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536852,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536974,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
537065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
537463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580986,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We can see that, there are a lot of zeros in the data but we also need to make sure any positive values are converted to a 1 and anything less the 0 is set to 0.

To know more about one hot encoding click [here](https://cdn.iisc.talentsprint.com/CDS/Assignments/Module6/M6_Supplementary_NB_2_Market_Basket_Analysis_LabelEncoding%20and%20One-Hot-Encoder.pdf)

In [21]:
def hot_encode(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1


basket_France = basket_France.applymap(hot_encode)


/var/folders/x_/586jm8gs3s385026b8mq6gmc0000gn/T/ipykernel_12915/1504403328.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_France = basket_France.applymap(hot_encode)


Now let's generate frequent item sets that have a support of at least 5%

In [22]:
frq_items = apriori(basket_France, min_support=0.05, use_colnames=True)


/Users/rajiv-ranjan/Documents/github/rajiv-ranjan/cds-mini-projects/m8/mp1/.venv/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Now let's generate the rules with their corresponding support, confidence and lift

In [23]:
rules = association_rules(frq_items, metric="lift", min_threshold=1)
rules = rules.sort_values(["confidence", "lift"], ascending=[False, False])
print(rules.head())


                                          antecedents  \
94  (SET/20 RED RETROSPOT PAPER NAPKINS , SET/6 RE...   
95  (SET/20 RED RETROSPOT PAPER NAPKINS , SET/6 RE...   
81                    (SET/6 RED SPOTTY PAPER PLATES)   
15                      (CHILDRENS CUTLERY SPACEBOY )   
48                     (PACK OF 6 SKULL PAPER PLATES)   

                        consequents  antecedent support  consequent support  \
94  (SET/6 RED SPOTTY PAPER PLATES)            0.105820            0.132275   
95    (SET/6 RED SPOTTY PAPER CUPS)            0.105820            0.142857   
81    (SET/6 RED SPOTTY PAPER CUPS)            0.132275            0.142857   
15  (CHILDRENS CUTLERY DOLLY GIRL )            0.071429            0.074074   
48     (PACK OF 6 SKULL PAPER CUPS)            0.058201            0.066138   

     support  confidence       lift  representativity  leverage  conviction  \
94  0.103175    0.975000   7.371000               1.0  0.089177   34.708995   
95  0.103175    0.975000